In [2]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import sympy as sy

In [ ]:
d, e1, e2, e3, u, w1, w2, w3, Q, omg, c = sy.symbols('d e1 e2 e3 u w1 w2 w3 Q omg c')
r12 = (e2*w1-e1*w2)/(e2*w1+e1*w2)
r23 = (e3*w2-e2*w3)/(e3*w2+e2*w3)
s11 = 1

list

In [2]:
n_prism = complex(1.531, -0.002)                   # комплексный показатель преломления призмы (Zeonex)
eps_prism = n_prism**2                             # диэлектрическая проницаемость призмы (Zeonex)
eps_air = n = 1                                    # диэлектрическая проницаемость воздуха (1.00027717)
eps_InSb = complex(-17.3, 7.12)                    # InSb на 197мкм из Telegram
wavelength = 197 * 10**(-6)                        # длина волны лазера в метрах
d_layer = 35 * 10**(-9)                            # толщина графеновой плёнки в метрах
step = 0.083*10**(-6)                              # шаг в метрах 

# -18.474508935284483+0.006568625288807081j - InSb по Друде-Лоренцу
# угол и зазор подгоняется из условия резонанса, т.к. гониометр и "линейка" неточные (???)
# какая точность нужна для расчётов? 

In [ ]:
def reflect3(gap_arr, angle, re_InSb, im_InSb, eps_prism, eps_air, wavelength):
    eps_InSb = np.complex128(complex(re_InSb, im_InSb))
    
    # Wave vectors in each medium
    kz_prism = np.sqrt(np.complex128(eps_prism)) * np.cos(angle)
    kz_air = np.sqrt(np.complex128(eps_air - eps_prism * np.sin(angle)**2))
    kz_3 = np.sqrt(np.complex128(eps_InSb - eps_prism * np.sin(angle)**2))
    
    # Reflection coefficients
    r_12 = (eps_air * kz_prism - eps_prism * kz_air) / (eps_air * kz_prism + eps_prism * kz_air)
    r_23 = (eps_InSb * kz_air - eps_air * kz_3) / (eps_InSb * kz_air + eps_air * kz_3)

    # Transmission coefficients
    t_12 = 2 * kz_prism * np.sqrt(np.complex128(eps_prism * eps_air)) / (eps_air * kz_prism + eps_prism * kz_air)
    t_23 = 2 * kz_air * np.sqrt(np.complex128(eps_air * eps_InSb)) / (eps_InSb * kz_air + eps_air * kz_3)

    # Precompute inverse of t_12 to avoid repeated division
    inv_t_12 = np.complex128(1 / t_12)
    inv_t_23 = np.complex128(1 / t_23)

    # Transmission matrix S1
    factor = r_12 * inv_t_12
    S1 = np.array([[inv_t_12, factor], [factor, inv_t_12]])
    
    # Reflectance calculation using vectorized operations
    phase_factor = np.exp(-1j * (2 * np.pi / wavelength) * kz_air * gap_arr)
    #print(phase_factor, '\n')
    S2_11 = phase_factor * inv_t_23
    S2_12 = r_23 * phase_factor * inv_t_23
    S2_21 = r_23 * np.conj(phase_factor) * inv_t_23
    S2_22 = np.conj(phase_factor) / inv_t_23
    S11 = S1[0, 0] * S2_11 + S1[0, 1] * S2_21
    S21 = S1[1, 0] * S2_11 + S1[1, 1] * S2_21
    R = np.abs(S21 / S11)
    
    return R